# VoiceForge — Colab GPU FastAPI Backend

This notebook runs the VoiceForge FastAPI server powered by **Qwen3-TTS VoiceDesign 1.7B** on Colab GPU with **Ngrok** tunneling.

### Endpoints:
- `GET /`: Health check
- `POST /generate`: Single text-to-speech generation (original VoiceForge endpoint)
- `POST /generate-document`: Document/PDF-to-audiobook generation with Groq narration rewriting, voice resolution, chunking, and audio concatenation.

### Secrets Setup:
In the Google Colab left sidebar (key icon 🔑), add:
1. `GROQ_API_KEY`: Your Groq API key
2. `NGROK_AUTHTOKEN`: Your Ngrok auth token

In [ ]:
# ── Cell 1: Install Dependencies ──────────────────────────────────
!pip install -U qwen-tts -q
!pip install fastapi uvicorn pyngrok nest-asyncio soundfile pypdf groq python-multipart -q

import torch
print(f"✅ CUDA available : {torch.cuda.is_available()}")
print(f"✅ GPU name       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
# ── Cell 2: Load Models ───────────────────────────────────────────
import os, torch
from qwen_tts import Qwen3TTSModel
from groq import Groq
from google.colab import userdata

print("Loading Qwen3-TTS VoiceDesign 1.7B model...")
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0" if torch.cuda.is_available() else "cpu",
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    attn_implementation="sdpa",
)
print("✅ Qwen3-TTS Model loaded successfully!")

# Groq client from Secrets
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
if groq_client:
    print("✅ Groq Client initialized successfully!")
else:
    print("⚠️ Warning: GROQ_API_KEY not found in Colab Secrets.")

In [ ]:
# ── Cell 3: FastAPI Server ────────────────────────────────────────
import os, io, re, json, base64, wave, struct, math
from fastapi import FastAPI, HTTPException, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pypdf import PdfReader
import soundfile as sf
import nest_asyncio

nest_asyncio.apply()

PLATFORM = "colab"

app = FastAPI(title=f"VoiceForge Studio API — {PLATFORM.upper()}")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Supported Languages
SUPPORTED_LANGUAGES = [
    "English", "Chinese", "Japanese", "Korean",
    "German", "French", "Russian", "Portuguese",
    "Spanish", "Italian"
]

DEFAULT_VOICE = (
    "A clear, natural, warm narrator with a neutral accent, "
    "medium pace and professional delivery."
)

GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-120b",
    "llama-3.1-8b-instant",
    "openai/gpt-oss-20b",
    "qwen/qwen3.8-27b"
]

class GenerateRequest(BaseModel):
    text: str
    instruct: str
    language: str = "English"

class GenerateResponse(BaseModel):
    audio_b64: str
    sample_rate: int
    platform: str
    duration_seconds: float

class GenerateDocumentResponse(BaseModel):
    audio_b64: str
    sample_rate: int
    platform: str
    duration_seconds: float
    voice_instruction: str
    num_chunks: int

def prepare_narration_with_groq(text: str) -> str:
    if not groq_client:
        return text.strip()
    prompt = f"""You are an expert audio scriptwriter.
Transform the following document text into a continuous, natural spoken narration script for an audiobook.
- Remove all page numbers, references, section headers, and formatting artifacts.
- Keep the narrative engaging, smooth, and comprehensive.
- Return ONLY the clean narration script text.

Text:
{text.strip()}
"""
    for m in GROQ_MODELS:
        try:
            res = groq_client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=m
            )
            narration = res.choices[0].message.content.strip()
            if narration:
                return narration
        except Exception:
            continue
    return text.strip()

def resolve_voice_profile(full_narration_text: str, user_instruct: str = "") -> str:
    if user_instruct and user_instruct.strip():
        return user_instruct.strip()
    if not groq_client:
        return DEFAULT_VOICE
    prompt = f"""You are a voice director. Based on the following complete audiobook narration script, determine the single ideal narrator voice profile for the entire reading.
Return ONLY a JSON object with fields: voice_type, gender, age, accent, tone, emotion, pace, style, instruction.
The 'instruction' field should be a single-sentence voice description suitable for a TTS model.

Script:
{full_narration_text[:1200]}
"""
    for m in GROQ_MODELS:
        try:
            res = groq_client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=m,
                response_format={"type": "json_object"}
            )
            data = json.loads(res.choices[0].message.content)
            instruction = data.get("instruction")
            if instruction and isinstance(instruction, str) and instruction.strip():
                return instruction.strip()
        except Exception:
            continue
    return DEFAULT_VOICE

def split_narration_into_chunks(text: str, max_chars: int = 400):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, current_chunk, current_len = [], [], 0
    for s in sentences:
        s = s.strip()
        if not s: continue
        if current_len + len(s) > max_chars and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = [s]
            current_len = len(s)
        else:
            current_chunk.append(s)
            current_len += len(s) + 1
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

def concatenate_wav_bytes(wav_bytes_list, pause_duration=0.3):
    if not wav_bytes_list:
        return b"", 24000, 0.0
    with wave.open(io.BytesIO(wav_bytes_list[0]), "rb") as first_wav:
        params = first_wav.getparams()
        sr, sw, nc = params.framerate, params.sampwidth, params.nchannels
    silence_frames = bytearray(int(sr * pause_duration) * sw * nc)
    out_buf, total_frames = io.BytesIO(), 0
    with wave.open(out_buf, "wb") as outfile:
        outfile.setparams(params)
        for idx, w_bytes in enumerate(wav_bytes_list):
            with wave.open(io.BytesIO(w_bytes), "rb") as infile:
                outfile.writeframes(infile.readframes(infile.getnframes()))
                total_frames += infile.getnframes()
            if idx < len(wav_bytes_list) - 1:
                outfile.writeframes(silence_frames)
                total_frames += int(sr * pause_duration)
    return out_buf.getvalue(), sr, total_frames / sr

@app.get("/")
def health():
    return {
        "status": "ok",
        "platform": PLATFORM,
        "model": "Qwen3-TTS-12Hz-1.7B-VoiceDesign",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "supported_languages": SUPPORTED_LANGUAGES,
    }

@app.post("/generate", response_model=GenerateResponse)
def generate_voice(req: GenerateRequest):
    if not req.text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty.")
    if len(req.text) > 500:
        raise HTTPException(status_code=400, detail="Text too long. Max 500 characters.")
    if not req.instruct.strip():
        raise HTTPException(status_code=400, detail="Instruct cannot be empty.")
    if len(req.instruct) > 300:
        raise HTTPException(status_code=400, detail="Instruct too long. Max 300 characters.")
    if req.language not in SUPPORTED_LANGUAGES:
        raise HTTPException(status_code=400, detail=f"Unsupported language. Choose from: {", ".join(SUPPORTED_LANGUAGES)}")

    try:
        wavs, sr = model.generate_voice_design(text=req.text, language=req.language, instruct=req.instruct)
        audio_array = wavs[0]
        buf = io.BytesIO()
        sf.write(buf, audio_array, sr, format="WAV")
        return GenerateResponse(
            audio_b64=base64.b64encode(buf.getvalue()).decode("utf-8"),
            sample_rate=sr,
            platform=PLATFORM,
            duration_seconds=round(len(audio_array) / sr, 2),
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Generation failed: {str(e)}")

@app.post("/generate-document", response_model=GenerateDocumentResponse)
async def generate_document(
    file: UploadFile = File(...),
    instruct: str = Form(""),
    language: str = Form("English")
):
    if language not in SUPPORTED_LANGUAGES:
        raise HTTPException(status_code=400, detail=f"Unsupported language. Choose from: {", ".join(SUPPORTED_LANGUAGES)}")

    try:
        content = await file.read()
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Failed to read uploaded file: {str(e)}")

    if not content or len(content) == 0:
        raise HTTPException(status_code=400, detail="PDF file is empty (0 bytes).")
    if not content.startswith(b"%PDF"):
        raise HTTPException(status_code=400, detail="Invalid PDF file. Header missing %PDF signature.")

    try:
        reader = PdfReader(io.BytesIO(content))
        raw_text = "\n\n".join([p.extract_text().strip() for p in reader.pages if p.extract_text() and p.extract_text().strip()])
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Invalid PDF extraction: {str(e)}")

    if not raw_text.strip():
        raise HTTPException(status_code=400, detail="No extractable text found in PDF.")

    narration_script = prepare_narration_with_groq(raw_text)
    unified_voice = resolve_voice_profile(narration_script, instruct)
    chunks = split_narration_into_chunks(narration_script, max_chars=400) or [narration_script]

    wav_segments = []
    try:
        for chunk in chunks:
            wavs, sr = model.generate_voice_design(text=chunk, language=language, instruct=unified_voice)
            buf = io.BytesIO()
            sf.write(buf, wavs[0], sr, format="WAV")
            wav_segments.append(buf.getvalue())
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"TTS synthesis failed: {str(e)}")

    combined_wav, final_sr, duration = concatenate_wav_bytes(wav_segments, pause_duration=0.3)
    return GenerateDocumentResponse(
        audio_b64=base64.b64encode(combined_wav).decode("utf-8"),
        sample_rate=final_sr,
        platform=PLATFORM,
        duration_seconds=round(duration, 2),
        voice_instruction=unified_voice,
        num_chunks=len(chunks)
    )

In [ ]:
# ── Cell 4: Start Ngrok + Uvicorn ────────────────────────────────
from pyngrok import ngrok
from google.colab import userdata
import uvicorn, asyncio, threading, time

# Load Ngrok token from Colab Secrets
ngrok_token = userdata.get("NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)
ngrok.kill()

tunnel = ngrok.connect(8000, bind_tls=True)

print("=" * 60)
print(f"  ✅ COLAB API URL        : {tunnel.public_url}")
print(f"  🔍 Health Check         : {tunnel.public_url}/")
print(f"  🎙️  Generate Voice       : {tunnel.public_url}/generate")
print(f"  📚 Generate Document    : {tunnel.public_url}/generate-document")
print("=" * 60)
print("  ⚠️  Copy this URL into your frontend SERVERS config!")
print("=" * 60)

def heartbeat():
    while True:
        time.sleep(3000)
        print("♥ Colab heartbeat")
threading.Thread(target=heartbeat, daemon=True).start()

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()